# Kaggle Inference Test - Modal H100 Full Checkpoint

Notebook ini khusus untuk inferensi (bukan training) dan memakai FULL checkpoint dari Hugging Face.

Alur notebook:
1. Clone repo + setup venv
2. Install dependencies
3. Download checkpoint FULL (dan distill sebagai referensi)
4. Download reference audio ref_prabowo.mp3
5. Jalankan inferensi dengan FULL checkpoint
6. Putar hasil audio

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from IPython.display import Audio, display

# ================= User Config =================
REPO_URL = "https://github.com/AneKazek/malesbgt.git"
REPO_BRANCH = "main"

# Semua artifact besar disimpan di /kaggle/temp
WORKDIR = Path("/kaggle/temp")
# Hanya hasil akhir inferensi disimpan ke /kaggle/working
RESULT_DIR = Path("/kaggle/working")
WORKDIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = WORKDIR / "kcv-tts"
VENV_DIR = REPO_DIR / ".venv"
VENV_PY = VENV_DIR / "bin/python"

HF_CKPT_REPO = "anekazek/kcv-tts-modal-h100-ckpt-20260402-125417"
HF_FULL_CKPT_FILENAME = "checkpoints/full/model_last.pt"
HF_DISTILL_CKPT_FILENAME = "checkpoints/distill/model_last.pt"

# Hardcoded token sesuai permintaan (tanpa env).
HF_TOKEN = "hf_xxx_ganti_dengan_token_kamu"

HF_DOWNLOAD_DIR = WORKDIR / "hf_downloads"
FULL_CKPT_LOCAL = HF_DOWNLOAD_DIR / HF_FULL_CKPT_FILENAME
DISTILL_CKPT_LOCAL = HF_DOWNLOAD_DIR / HF_DISTILL_CKPT_FILENAME

REF_REPO_ID = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
REF_FILENAME = "ref_prabowo.mp3"
REF_TEXT = "SELAMAT PAGI SALAM SEJAHTERA BAGI KITA SEMUA"
GEN_TEXT = REF_TEXT
REF_AUDIO_LOCAL = HF_DOWNLOAD_DIR / REF_FILENAME

VOCAB_PATH = REPO_DIR / "data/Emilia_ZH_EN_pinyin/vocab.txt"
INFER_OUT_WAV = RESULT_DIR / "infer_modal_h100_full_ckpt.wav"

# Sesuaikan jika arsitektur checkpoint full kamu berbeda.
MODEL_NAME = "F5TTS_EarlyBiMamba_v1"

def run_cmd(cmd, cwd=None, env=None, timeout=None):
    printable = cmd if isinstance(cmd, str) else " ".join(str(x) for x in cmd)
    print("\n$", printable)
    if timeout is not None:
        print(f"(timeout={timeout}s)")
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        timeout=timeout,
    )

def run_py(args, cwd=None, env=None, timeout=None):
    return run_cmd(["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args], cwd=cwd, env=env, timeout=timeout)

print("Config siap.")
print("Runtime dir:", WORKDIR)
print("Result dir :", RESULT_DIR)
print("Repo dir:   ", REPO_DIR)
print("Full ckpt:  ", FULL_CKPT_LOCAL)
print("Distill ckpt:", DISTILL_CKPT_LOCAL)
print("Ref audio:  ", REF_AUDIO_LOCAL)
print("Infer out:  ", INFER_OUT_WAV)
print("HF token hardcoded:", bool(HF_TOKEN and HF_TOKEN != "hf_xxx_ganti_dengan_token_kamu"))
if HF_TOKEN == "hf_xxx_ganti_dengan_token_kamu":
    print("Ganti HF_TOKEN dengan token asli kamu di Cell 2.")

In [ ]:
# 1) Clone repo
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_cmd(["git", "clone", "--recursive", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])
run_cmd(["ls", "-lah", str(REPO_DIR)])

In [ ]:
# 2) Setup venv + dependencies
if shutil.which("uv") is None:
    run_cmd(["python3", "-m", "pip", "install", "-U", "uv"])

TARGET_PY_MM = "3.11"
TARGET_PY = Path(f"/usr/bin/python{TARGET_PY_MM}")

run_cmd(["apt-get", "update", "-y"])
run_cmd([
    "apt-get",
    "install",
    "-y",
    f"python{TARGET_PY_MM}",
    f"python{TARGET_PY_MM}-venv",
    f"python{TARGET_PY_MM}-dev",
    "build-essential",
])

if not TARGET_PY.exists():
    raise FileNotFoundError(f"Interpreter target tidak ditemukan: {TARGET_PY}")

run_cmd(["uv", "venv", "--python", str(TARGET_PY), "--clear", str(VENV_DIR)])
run_cmd([str(VENV_PY), "-c", "import sys; print('venv python =', sys.version)"])

run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--upgrade",
    "pip",
    "wheel",
    "setuptools<82",
])

req_candidates = [
    REPO_DIR / "requirements-torch28-cu12-localmatch.txt",
    REPO_DIR / "requirements-kaggle-torch210.txt",
]
REQ_PROFILE = next((p for p in req_candidates if p.exists()), None)
if REQ_PROFILE is None:
    raise FileNotFoundError("Tidak menemukan requirements profile untuk setup inference.")

print("Using dependency profile:", REQ_PROFILE)
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--index-strategy",
    "unsafe-best-match",
    "-r",
    str(REQ_PROFILE),
])

run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "-e",
    str(REPO_DIR),
], cwd=REPO_DIR)

run_cmd([
    str(VENV_PY),
    "-c",
    "import torch, f5_tts; print('torch =', torch.__version__, 'cuda =', torch.version.cuda); print('f5_tts import ok')",
])

In [ ]:
# 3) Download FULL checkpoint dulu (wajib), plus distill (opsional referensi) dan reference audio
run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"], cwd=REPO_DIR)

if not HF_TOKEN or HF_TOKEN == "hf_xxx_ganti_dengan_token_kamu":
    raise ValueError("HF_TOKEN masih placeholder. Isi token asli di Cell 2 dulu.")

download_script = "\n".join([
    "from pathlib import Path",
    "from huggingface_hub import hf_hub_download",
    f"token = {HF_TOKEN!r}",
    "print('Using hardcoded HF token for checkpoint download')",
    f"repo_id = {HF_CKPT_REPO!r}",
    f"full_filename = {HF_FULL_CKPT_FILENAME!r}",
    f"distill_filename = {HF_DISTILL_CKPT_FILENAME!r}",
    f"ref_repo_id = {REF_REPO_ID!r}",
    f"ref_filename = {REF_FILENAME!r}",
    f"download_root = Path({str(HF_DOWNLOAD_DIR)!r})",
    "download_root.mkdir(parents=True, exist_ok=True)",
    "def dl(repo_id, filename, out_dir):",
    "    return Path(hf_hub_download(",
    "        repo_id=repo_id,",
    "        filename=filename,",
    "        local_dir=str(out_dir),",
    "        token=token,",
    "    ))",
    "try:",
    "    full_target = dl(repo_id, full_filename, download_root)",
    "    distill_target = dl(repo_id, distill_filename, download_root)",
    "except Exception as e:",
    "    raise RuntimeError(f'Gagal download checkpoint: {type(e).__name__}: {e}') from e",
    "ref_target = dl(ref_repo_id, ref_filename, download_root)",
    "print('FULL ckpt   :', full_target, 'size=', full_target.stat().st_size)",
    "print('DISTILL ckpt:', distill_target, 'size=', distill_target.stat().st_size)",
    "print('REF audio   :', ref_target, 'size=', ref_target.stat().st_size)",
])
run_py(["-c", download_script], cwd=REPO_DIR)

# Pastikan vocab tersedia untuk API F5TTS
vocab_script = "\n".join([
    "from pathlib import Path",
    "import shutil",
    "from huggingface_hub import hf_hub_download",
    f"token = {HF_TOKEN!r}",
    f"target = Path({str(VOCAB_PATH)!r})",
    "target.parent.mkdir(parents=True, exist_ok=True)",
    "if target.exists() and target.stat().st_size > 0:",
    "    print('vocab already exists:', target)",
    "else:",
    "    candidates = [('SWivid/F5-TTS', 'F5TTS_Base/vocab.txt'), ('SWivid/F5-TTS', 'F5TTS_v1_Base/vocab.txt')]",
    "    last_err = None",
    "    for repo_id, filename in candidates:",
    "        try:",
    "            src = Path(hf_hub_download(repo_id=repo_id, filename=filename, token=token))",
    "            shutil.copy2(src, target)",
    "            print(f'vocab downloaded from {repo_id}/{filename} -> {target}')",
    "            break",
    "        except Exception as e:",
    "            last_err = e",
    "    else:",
    "        raise RuntimeError(f'Gagal download vocab: {last_err}')",
    "print('vocab:', target, 'size=', target.stat().st_size)",
])
run_py(["-c", vocab_script], cwd=REPO_DIR)

In [ ]:
# 4) Inference pakai FULL checkpoint (dtype-safe untuk Mamba)
runtime_env = os.environ.copy()
runtime_env["MPLBACKEND"] = "Agg"
if HF_TOKEN and HF_TOKEN != "hf_xxx_ganti_dengan_token_kamu":
    runtime_env["HF_TOKEN"] = HF_TOKEN

infer_lines = [
    "from pathlib import Path",
    "from f5_tts.api import F5TTS",
    "import torch",
    f"model_name = {MODEL_NAME!r}",
    f"ckpt_file = Path({str(FULL_CKPT_LOCAL)!r})",
    f"vocab_file = Path({str(VOCAB_PATH)!r})",
    f"ref_audio = Path({str(REF_AUDIO_LOCAL)!r})",
    f"out_wav = Path({str(INFER_OUT_WAV)!r})",
    f"ref_text = {REF_TEXT!r}",
    f"gen_text = {GEN_TEXT!r}",
    "if not ckpt_file.exists():",
    "    raise FileNotFoundError(f'Checkpoint full tidak ditemukan: {ckpt_file}')",
    "if not vocab_file.exists():",
    "    raise FileNotFoundError(f'Vocab tidak ditemukan: {vocab_file}')",
    "if not ref_audio.exists():",
    "    raise FileNotFoundError(f'Reference audio tidak ditemukan: {ref_audio}')",
    "device = 'cuda' if torch.cuda.is_available() else 'cpu'",
    "print('device =', device)",
    "print('model  =', model_name)",
    "print('ckpt   =', ckpt_file)",
    "tts = F5TTS(",
    "    model=model_name,",
    "    ckpt_file=str(ckpt_file),",
    "    vocab_file=str(vocab_file),",
    "    use_ema=True,",
    "    device=device,",
    ")",
    "# Penting: hindari .half() paksa, karena bisa memicu dtype mismatch di jalur Mamba.",
    "if hasattr(tts, 'ema_model') and tts.ema_model is not None:",
    "    tts.ema_model.float()",
    "if hasattr(tts, 'model') and tts.model is not None:",
    "    tts.model.float()",
    "tts.infer(",
    "    ref_file=str(ref_audio),",
    "    ref_text=ref_text,",
    "    gen_text=gen_text,",
    "    file_wave=str(out_wav),",
    "    nfe_step=32,",
    "    speed=1.0,",
    "    seed=1234,",
    ")",
    "print('Inference selesai')",
    "print('out =', out_wav)",
]

infer_script = "\n".join(infer_lines)
run_py(["-c", infer_script], cwd=REPO_DIR, env=runtime_env)

In [ ]:
# 5) Play output
if not INFER_OUT_WAV.exists():
    raise FileNotFoundError(f"Output audio tidak ditemukan: {INFER_OUT_WAV}")

print("Saved:", INFER_OUT_WAV)
display(Audio(str(INFER_OUT_WAV)))